# Carbon Mapper Interactive Dashboard - Thesis Stage 0

This notebook launches an interactive dashboard for querying and exploring
high-resolution Carbon Mapper CH4 and CO2 plume observations. It is the Stage 0
data-access and exploratory-analysis component of the thesis workflow.

The notebook contains no Sentinel-5P/TROPOMI processing or Earth Engine
matching. Carbon Mapper is the only external plume-data source used by the
dashboard.


## What this notebook does

The dashboard allows the user to:

1. select a country and optional states/provinces;
2. select a date range and gas type, CH₄ and/or CO₂;
3. fetch matching plume records from the Carbon Mapper API;
4. optionally estimate plume area from available Carbon Mapper raster links;
5. summarize plume counts, plume areas, and available emission fields by administrative region;
6. visualize results using bar charts, histograms, choropleth maps, plume points, and optional heatmaps;
7. export or download the raw Carbon Mapper CSV queried by the app.

The implementation lives in the companion Python module:

```text
carbon_mapper_dashboard.py
```

The notebook imports the companion module from the same directory.


In [ ]:
# Install the Python packages required by the Carbon Mapper dashboard.
# In Colab this cell is usually needed once per runtime.
# In a local environment, you can instead install these packages in your virtual environment.

!pip -q install geopandas rasterio folium ipywidgets shapely pandas numpy requests matplotlib

# Enable ipywidgets rendering in Google Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    # This notebook can also run outside Colab, where google.colab is unavailable.
    pass


## Locate the companion Python module

The next cell searches for `carbon_mapper_dashboard.py`, adds its folder to `sys.path`, and then imports the dashboard class.

The module search includes the notebook directory, the Colab runtime, and mounted Google Drive locations.


In [ ]:
import os
import sys
from pathlib import Path

MODULE_FILE = "carbon_mapper_dashboard.py"

# Mount Google Drive if running in Colab. This is useful when the repository is stored in Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

candidate_paths = []

# 1) Current working directory
candidate_paths.append(Path.cwd() / MODULE_FILE)

# 2) Common Colab upload location
candidate_paths.append(Path("/content") / MODULE_FILE)

# 3) Common thesis folder location in Drive
candidate_paths.append(Path("/content/drive/MyDrive/thesis") / MODULE_FILE)

# 4) Search MyDrive recursively if available
mydrive = Path("/content/drive/MyDrive")
if mydrive.exists():
    candidate_paths.extend(mydrive.glob(f"**/{MODULE_FILE}"))

module_path = next((p for p in candidate_paths if p.exists()), None)

if module_path is None:
    raise FileNotFoundError(
        f"Could not find {MODULE_FILE}. Upload it to /content, place it beside this notebook, "
        "or update the search path in this cell."
    )

module_dir = str(module_path.parent)
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

print(f"Using dashboard module: {module_path}")

from carbon_mapper_dashboard import CMApp


## Carbon Mapper API authentication

The API token is requested through hidden input and stored only in the current
runtime environment variable `CM_TOKEN`. It is not written into the notebook
source or saved outputs.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("CM_TOKEN"):
    os.environ["CM_TOKEN"] = getpass("Paste your Carbon Mapper API token: ").strip()

if not os.environ.get("CM_TOKEN"):
    raise ValueError("CM_TOKEN is empty. Please provide a valid Carbon Mapper API token.")

print("Carbon Mapper token is set for this runtime session.")


## Launch the dashboard

An initial validation run can use the following configuration:

- Country: `United States of America`
- Date range: one month
- Gas: `CH4`
- Maximum records: `500`
- States/provinces: empty for the full selected country

Larger date ranges and regional filters can be applied after the initial run.


In [ ]:
app = CMApp(cm_token=os.environ["CM_TOKEN"])
app.display()


## Optional: export state/province statistics after running the dashboard

The helper method below exports summary statistics after **Fetch & Analyze** has loaded data. The `gas` argument selects CH4 or CO2 records.


In [ ]:
# Optional export after plume data have been loaded.
# app.export_state_stats(path="carbon_mapper_state_stats_ch4.csv", gas="CH4")


## Optional: save the interactive map as HTML

After **Fetch & Analyze** has loaded data, the current map can be saved as an HTML supplementary artifact.


In [ ]:
# Optional map export after plume data have been loaded.
# app.save_map(path="carbon_mapper_ch4_map.html", gas="CH4", metric="plume_count")


## Scientific interpretation

The dashboard queries Carbon Mapper plume metadata for a selected country,
administrative region, date interval, and gas type. It summarizes and
visualizes plume occurrence and available plume attributes by administrative
unit. Optional plume-area estimation uses linked Carbon Mapper raster products
for exploratory comparison.

This stage is a data-access, visualization, and exploratory-analysis tool. It
is not an emissions validation system.
